[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/transferibilidad_03_explicabilidad_cnn.ipynb)

# Landslide4Sense — Explicabilidad CNN: Grad-CAM

> **Notebook 3/3 — Transferibilidad a Colombia**  
> Visualiza qué regiones espaciales activan ResNet-50 y EfficientNet-B4.  
> Si los checkpoints entrenados están en Drive, se cargan automáticamente.  
> Sin checkpoints, se usan pesos ImageNet — las activaciones son indicativas, no definitivas.

**Requiere GPU (T4/A100)** — activar en *Runtime → Change runtime type*.

## 0 · Entorno y dependencias

In [ ]:
import importlib, subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, mod in [('grad-cam', 'pytorch_grad_cam'), ('timm', 'timm'), ('h5py', 'h5py')]:
    try:
        importlib.import_module(mod)
    except ModuleNotFoundError:
        pip_install(pkg)
print('Dependencias listas.')

## 1 · Google Drive y rutas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# ── Ajusta al directorio raíz de tu proyecto en Drive ──────────────
DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'

ROOT     = Path(DRIVE_PATH)
IMG_DIR  = ROOT / 'TrainData' / 'img'    # directorio con .h5 individuales
MASK_DIR = ROOT / 'TrainData' / 'mask'   # directorio con .h5 de máscara
OUT_DIR  = ROOT / 'results' / 'gradcam'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoints entrenados (generados por run_training.py) ─────────────
CKPT_RN50  = ROOT / 'results' / 'resnet50'       / 'fold_0' / 'best_model.pth'
CKPT_EFFB4 = ROOT / 'results' / 'efficientnet_b4' / 'fold_0' / 'best_model.pth'

# Verificación ───────────────────────────────────────────────────────
img_files = sorted(IMG_DIR.glob('*.h5'))
print(f'Imágenes encontradas : {len(img_files)}')
print(f'Directorio de salida : {OUT_DIR}')
print()
for label, ckpt in [('ResNet-50', CKPT_RN50), ('EfficientNet-B4', CKPT_EFFB4)]:
    status = '✅' if ckpt.exists() else '⚠️  no encontrado — se usarán pesos ImageNet'
    print(f'{label:20s}: {status}')
    if not ckpt.exists():
        print(f'    Ruta esperada: {ckpt}')

## 2 · Importaciones

In [ ]:
import numpy as np
import h5py
import random
import matplotlib.pyplot as plt
import cv2  # colormap entero para pytorch_grad_cam
import torch
import torch.nn as nn
import torchvision.models as tv_models
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import timm
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Dispositivo: {DEVICE}')
if DEVICE == 'cpu':
    print('⚠️  Sin GPU — Grad-CAM funcionará pero lento. Activa T4 en Runtime.')

## 3 · Carga de datos (misma estructura que NB1/NB2)

In [ ]:
N_SAMPLES = 400   # parches a cargar (aumenta si tienes tiempo/RAM)

def load_dataset(n=N_SAMPLES, seed=SEED):
    img_files = sorted(IMG_DIR.glob('*.h5'))
    rng = np.random.default_rng(seed)
    chosen = sorted(rng.choice(len(img_files), min(n, len(img_files)), replace=False))
    patches, masks = [], []
    for i in chosen:
        with h5py.File(img_files[i], 'r') as f:
            key = list(f.keys())[0]
            patches.append(f[key][:].astype(np.float32))  # (128,128,14)
        mask_path = MASK_DIR / img_files[i].name
        if mask_path.exists():
            with h5py.File(mask_path, 'r') as f:
                key = list(f.keys())[0]
                masks.append(f[key][:].astype(np.float32))  # (128,128)
        else:
            masks.append(np.zeros((128,128), dtype=np.float32))
    return np.stack(patches), np.stack(masks)

patches_raw, masks_raw = load_dataset()
N = patches_raw.shape[0]
labels = (masks_raw.reshape(N, -1).mean(axis=1) > 0.02).astype(int)
print(f'Parches: {N}  |  Positivos: {labels.sum()}  |  Negativos: {(labels==0).sum()}')
print(f'Shape parches: {patches_raw.shape}  |  Shape máscaras: {masks_raw.shape}')

## 3b · (Opcional) Fine-tuning rápido para generar checkpoints

> Corre esta sección **solo si no tienes** `results/resnet50/fold_0/best_model.pth` en Drive.  
> Fine-tunea ResNet-50 durante **8 épocas** sobre los datos cargados (~10 min en T4).  
> El checkpoint resultante activa la alineación espacial Grad-CAM ↔ Máscara (§12).

Si ya tienes los checkpoints del entrenamiento completo, **salta esta celda**.

In [ ]:
# Constantes de preprocesamiento (definidas tambien en SS5)
RGB_IDX = [4, 5, 6]   # B8A, B11, B12 -> R, G, B
import numpy as _np
MEAN_IN = _np.array([0.485, 0.456, 0.406], dtype='float32')
STD_IN  = _np.array([0.229, 0.224, 0.225], dtype='float32')

import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.models as tv_models

# -- Parametros del mini-entrenamiento -----------------------
FINETUNE_EPOCHS = 8
BATCH_SIZE      = 32
LR_HEAD         = 1e-3
LR_BACKBONE     = 1e-4

# pos_weight calculado desde los datos reales
# Si hay mas positivos que negativos, forzamos pos_weight >= 1
# para que el modelo no colapse a predecir todo negativo
n_pos = int(labels.sum())
n_neg = int((labels == 0).sum())
POS_WEIGHT_VAL = max(1.0, n_neg / (n_pos + 1e-6))
print(f'Positivos: {n_pos}  |  Negativos: {n_neg}  |  pos_weight={POS_WEIGHT_VAL:.3f}')

def patches_to_tensors(patches, labels_arr, rgb_idx=RGB_IDX):
    out = []
    for p in patches:
        rgb = p[:, :, rgb_idx].astype(_np.float32)
        p2, p98 = _np.percentile(rgb, 2), _np.percentile(rgb, 98)
        rgb_vis = _np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
        rgb_norm = (rgb_vis - MEAN_IN) / STD_IN
        out.append(rgb_norm.transpose(2, 0, 1))
    X = torch.tensor(_np.stack(out), dtype=torch.float32)
    y = torch.tensor(labels_arr, dtype=torch.float32).unsqueeze(1)
    return X, y

# -- Split train/val -----------------------------------------
X_tr_idx, X_va_idx = train_test_split(
    _np.arange(N), test_size=0.2, stratify=labels, random_state=SEED)

X_tr, y_tr = patches_to_tensors(patches_raw[X_tr_idx], labels[X_tr_idx])
X_va, y_va = patches_to_tensors(patches_raw[X_va_idx], labels[X_va_idx])

dl_tr = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
dl_va = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f'Train: {len(X_tr)}  |  Val: {len(X_va)}')

# -- Modelo con bias inicializado para evitar colapso --------
ft_model = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
ft_model.fc = nn.Linear(ft_model.fc.in_features, 1)
# Bias inicial = log(p_pos / p_neg) acelera convergencia
bias_init = float(_np.log(n_pos / (n_neg + 1e-6)))
torch.nn.init.constant_(ft_model.fc.bias, bias_init)
ft_model = ft_model.to(DEVICE)

optimizer = optim.AdamW([
    {'params': ft_model.fc.parameters(),
     'lr': LR_HEAD},
    {'params': [p for nm, p in ft_model.named_parameters() if 'fc' not in nm],
     'lr': LR_BACKBONE}
], weight_decay=1e-4)

pos_w     = torch.tensor([POS_WEIGHT_VAL]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

# -- Bucle de entrenamiento ----------------------------------
from sklearn.metrics import f1_score
best_f1, best_state = 0.0, None

for epoch in range(1, FINETUNE_EPOCHS + 1):
    ft_model.train()
    tr_loss = 0.0
    for xb, yb in dl_tr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(ft_model(xb), yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item()

    ft_model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in dl_va:
            xb = xb.to(DEVICE)
            prob = torch.sigmoid(ft_model(xb)).cpu().numpy().flatten()
            preds.extend((prob >= 0.5).astype(int).tolist())
            trues.extend(yb.numpy().flatten().astype(int).tolist())
    f1       = f1_score(trues, preds, zero_division=0)
    pos_pred = sum(preds)
    print(f'Epoch {epoch:2d}/{FINETUNE_EPOCHS}  '
          f'loss={tr_loss/len(dl_tr):.4f}  '
          f'val_F1={f1:.3f}  '
          f'pred_pos={pos_pred}/{len(preds)}')
    if f1 >= best_f1:
        best_f1 = f1
        best_state = {k: v.cpu().clone() for k, v in ft_model.state_dict().items()}

# -- Guardar checkpoint --------------------------------------
ckpt_dir = CKPT_RN50.parent
ckpt_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_state, str(CKPT_RN50))
print(f'\nCheckpoint guardado: {CKPT_RN50}  (best val F1={best_f1:.3f})')
print('SS4 cargara este checkpoint automaticamente al construir rn50.')


## 4 · Construcción de modelos

Los modelos reciben 3 canales (RGB). Mapeamos desde los 14 canales L4S:
- **R** → índice 4 (S2 B8A – NIR-A)
- **G** → índice 5 (S2 B11 – SWIR1)
- **B** → índice 6 (S2 B12 – SWIR2)

Si el checkpoint entrenado existe se carga; si no, se usan pesos ImageNet  
y el Grad-CAM mostrará las activaciones del modelo base (no fine-tuned).

In [ ]:
RGB_IDX = [4, 5, 6]   # B8A, B11, B12 → R, G, B

def load_model_weights(model, ckpt_path, label):
    if ckpt_path.exists():
        state = torch.load(str(ckpt_path), map_location=DEVICE)
        # El checkpoint puede ser state_dict directo o {'model_state_dict': ...}
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'{label}: checkpoint cargado ✅  (missing={len(missing)}, unexpected={len(unexpected)})')
        return True
    else:
        print(f'{label}: checkpoint no encontrado — usando pesos ImageNet ⚠️')
        return False

# ResNet-50
rn50 = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
rn50.fc = nn.Linear(rn50.fc.in_features, 1)
ckpt_loaded_rn50 = load_model_weights(rn50, CKPT_RN50, 'ResNet-50')
rn50 = rn50.to(DEVICE).eval()

# EfficientNet-B4
effb4 = timm.create_model('efficientnet_b4', pretrained=True, num_classes=1)
ckpt_loaded_eff = load_model_weights(effb4, CKPT_EFFB4, 'EfficientNet-B4')
effb4 = effb4.to(DEVICE).eval()

## 5 · Preprocesamiento de parches

In [ ]:
MEAN_IN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD_IN  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_patch(patch_hw14, rgb_idx=RGB_IDX):
    """
    patch_hw14: (128,128,14) float32
    Retorna: tensor (1,3,128,128) en DEVICE  +  rgb_vis (128,128,3) en [0,1]
    """
    rgb = patch_hw14[:, :, rgb_idx].copy()       # (128,128,3)
    p2, p98 = np.percentile(rgb, 2), np.percentile(rgb, 98)
    rgb_vis = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1).astype(np.float32)
    rgb_norm = (rgb_vis - MEAN_IN) / STD_IN
    tensor = torch.tensor(rgb_norm.transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    return tensor, rgb_vis

# Test rápido
t_test, v_test = preprocess_patch(patches_raw[0])
print(f'Tensor: {t_test.shape}  dtype={t_test.dtype}  rango=[{t_test.min():.2f},{t_test.max():.2f}]')

## 6 · Helpers Grad-CAM

In [ ]:
def get_gradcam(model, tensor, target_layer):
    """Retorna heatmap (H,W) en [0,1]."""
    with GradCAM(model=model, target_layers=[target_layer]) as cam:
        heatmap = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(0)])[0]
    return np.float32(heatmap)

def overlay_cam(rgb_vis, heatmap, colormap=cv2.COLORMAP_JET, alpha=0.4):
    """Superpone el heatmap sobre rgb_vis (float32 en [0,1]).\n    colormap debe ser entero OpenCV (cv2.COLORMAP_*)."""
    return show_cam_on_image(rgb_vis, heatmap, use_rgb=True, colormap=colormap, image_weight=1-alpha)

# Capas objetivo para Grad-CAM
TGT_RN50 = rn50.layer4[-1]          # (2048 @ 4×4)
TGT_EFF  = effb4.conv_head           # capa conv final de EfficientNet-B4
print('Capas objetivo configuradas ✅')


## 7 · Selección de parches representativos

Estratificamos por cobertura de deslizamiento para cubrir casos variados.

In [ ]:
coverage = masks_raw.reshape(N, -1).mean(axis=1)
rng = np.random.default_rng(SEED)

grupos = [
    ('Pequeño',  (labels==1) & (coverage >  0)    & (coverage <  0.10), 2),
    ('Mediano',  (labels==1) & (coverage >= 0.10) & (coverage <  0.40), 2),
    ('Grande',   (labels==1) & (coverage >= 0.40),                       2),
    ('Negativo', labels==0,                                               2),
]

selected = []
for nombre, mask_g, n in grupos:
    pool = np.where(mask_g)[0]
    if len(pool) == 0:
        print(f'  ⚠️  Sin parches para grupo «{nombre}»')
        continue
    idxs = rng.choice(pool, min(n, len(pool)), replace=False)
    for i in idxs:
        selected.append({'idx': int(i), 'label': int(labels[i]),
                         'cov': float(coverage[i]), 'grupo': nombre})

print(f'Parches seleccionados: {len(selected)}')
for s in selected:
    print(f"  [{s['grupo']:8s}] idx={s['idx']:4d}  label={s['label']}  cov={s['cov']:.3f}")

## 8 · Grad-CAM — ResNet-50

> Capa `layer4[-1]` — receptive field de 4×4 → upsampled a 128×128.

In [ ]:
fig, axes = plt.subplots(len(selected), 3, figsize=(12, 3.2*len(selected)))
fig.suptitle(
    f'Grad-CAM — ResNet-50{"" if ckpt_loaded_rn50 else " (ImageNet, sin fine-tuning)"}',    fontsize=13, fontweight='bold')

for row, s in enumerate(selected):
    patch = patches_raw[s['idx']]
    mask  = masks_raw[s['idx']]
    tensor, rgb_vis = preprocess_patch(patch)

    with torch.no_grad():
        prob = torch.sigmoid(rn50(tensor)).item()

    heatmap = get_gradcam(rn50, tensor, TGT_RN50)
    overlay = overlay_cam(rgb_vis, heatmap)

    axes[row,0].imshow(rgb_vis)
    axes[row,0].set_title(f"{s['grupo']}  cov={s['cov']:.2f}  p={prob:.2f}", fontsize=9)
    axes[row,0].axis('off')
    axes[row,1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
    axes[row,1].set_title('Máscara real', fontsize=9)
    axes[row,1].axis('off')
    axes[row,2].imshow(overlay)
    axes[row,2].set_title(f'Grad-CAM  max={heatmap.max():.2f}', fontsize=9)
    axes[row,2].axis('off')

plt.tight_layout()
out_rn50 = OUT_DIR / 'gradcam_resnet50.png'
plt.savefig(out_rn50, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_rn50}')

## 9 · Grad-CAM — EfficientNet-B4

> Capa `conv_head` — mayor resolución espacial que ResNet-50.

In [ ]:
fig, axes = plt.subplots(len(selected), 3, figsize=(12, 3.2*len(selected)))
fig.suptitle(
    f'Grad-CAM — EfficientNet-B4{"" if ckpt_loaded_eff else " (ImageNet, sin fine-tuning)"}',    fontsize=13, fontweight='bold')

for row, s in enumerate(selected):
    patch = patches_raw[s['idx']]
    mask  = masks_raw[s['idx']]
    tensor, rgb_vis = preprocess_patch(patch)

    with torch.no_grad():
        prob = torch.sigmoid(effb4(tensor)).item()

    heatmap = get_gradcam(effb4, tensor, TGT_EFF)
    overlay = overlay_cam(rgb_vis, heatmap)

    axes[row,0].imshow(rgb_vis)
    axes[row,0].set_title(f"{s['grupo']}  cov={s['cov']:.2f}  p={prob:.2f}", fontsize=9)
    axes[row,0].axis('off')
    axes[row,1].imshow(mask, cmap='Reds', vmin=0, vmax=1)
    axes[row,1].set_title('Máscara real', fontsize=9)
    axes[row,1].axis('off')
    axes[row,2].imshow(overlay)
    axes[row,2].set_title(f'Grad-CAM  max={heatmap.max():.2f}', fontsize=9)
    axes[row,2].axis('off')

plt.tight_layout()
out_eff = OUT_DIR / 'gradcam_efficientnet_b4.png'
plt.savefig(out_eff, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_eff}')

## 10 · Comparación: ResNet-50 vs EfficientNet-B4 (parches medianos)

In [ ]:
med = [s for s in selected if s['grupo'] == 'Mediano']
if not med:
    med = selected[:2]   # fallback si no hay medianos

fig, axes = plt.subplots(len(med), 4, figsize=(16, 4*len(med)))
if len(med) == 1: axes = axes[np.newaxis, :]
fig.suptitle('Comparación Grad-CAM — ResNet-50 vs EfficientNet-B4', fontsize=13)

for row, s in enumerate(med):
    patch  = patches_raw[s['idx']]
    mask   = masks_raw[s['idx']]
    tensor, rgb_vis = preprocess_patch(patch)
    hm_rn  = get_gradcam(rn50,  tensor, TGT_RN50)
    hm_eff = get_gradcam(effb4, tensor, TGT_EFF)

    axes[row,0].imshow(rgb_vis);                    axes[row,0].set_title(f'RGB (idx={s["idx"]})', fontsize=9); axes[row,0].axis('off')
    axes[row,1].imshow(mask, cmap='Reds');          axes[row,1].set_title('Máscara', fontsize=9);               axes[row,1].axis('off')
    axes[row,2].imshow(overlay_cam(rgb_vis,hm_rn)); axes[row,2].set_title('ResNet-50', fontsize=9);             axes[row,2].axis('off')
    axes[row,3].imshow(overlay_cam(rgb_vis,hm_eff));axes[row,3].set_title('EfficientNet-B4', fontsize=9);       axes[row,3].axis('off')

plt.tight_layout()
out_cmp = OUT_DIR / 'gradcam_comparison.png'
plt.savefig(out_cmp, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_cmp}')

## 11 · Simulación Colombia — Grad-CAM SAR-solo

Enmascaramos los canales ópticos (0–6, 11–13) dejando solo SAR+DEM (7–10).  
Esto simula la situación de alta nubosidad (60–80%) en los Andes colombianos.

In [ ]:
OPTICAL_IDX = list(range(7)) + [11, 12, 13]

def mask_optical(patch):
    p = patch.copy()
    p[:, :, OPTICAL_IDX] = 0.0
    return p

med2 = [s for s in selected if s['grupo'] == 'Mediano'][:2] or selected[:2]

fig, axes = plt.subplots(len(med2), 4, figsize=(16, 4*len(med2)))
if len(med2) == 1: axes = axes[np.newaxis, :]
fig.suptitle('ResNet-50 Grad-CAM: Todos los canales vs SAR-solo (Colombia)', fontsize=13)

for row, s in enumerate(med2):
    patch_full = patches_raw[s['idx']]
    patch_sar  = mask_optical(patch_full)
    mask_arr   = masks_raw[s['idx']]

    tf, rgb_f = preprocess_patch(patch_full)
    ts, rgb_s = preprocess_patch(patch_sar)

    hm_f = get_gradcam(rn50, tf, TGT_RN50)
    hm_s = get_gradcam(rn50, ts, TGT_RN50)

    axes[row,0].imshow(rgb_f);                   axes[row,0].set_title('RGB completo',   fontsize=9); axes[row,0].axis('off')
    axes[row,1].imshow(mask_arr, cmap='Reds');   axes[row,1].set_title('Máscara',         fontsize=9); axes[row,1].axis('off')
    axes[row,2].imshow(overlay_cam(rgb_f,hm_f)); axes[row,2].set_title('Grad-CAM completo', fontsize=9); axes[row,2].axis('off')
    axes[row,3].imshow(overlay_cam(rgb_s,hm_s)); axes[row,3].set_title('Grad-CAM SAR-solo', fontsize=9); axes[row,3].axis('off')

plt.tight_layout()
out_sar = OUT_DIR / 'gradcam_sar_only.png'
plt.savefig(out_sar, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_sar}')

## 12 · Discriminacion espacial Grad-CAM (funciona sin checkpoint)

Medimos el **ratio de activacion dentro vs fuera** de la mascara real.  
> **Ratio > 1.0** = el modelo activa mas en la zona del deslizamiento (discriminativo).  
> **Ratio ~ 1.0** = activacion uniforme (modelo no entrenado en L4S).  
> Esta metrica funciona con cualquier peso, incluidos ImageNet sin fine-tuning.


In [ ]:
# Analisis de Grad-CAM que funciona con CUALQUIER peso (ImageNet o fine-tuned)
# Metrica: activacion media dentro de la mascara vs fuera (ratio de discriminacion)
# No requiere correlacion de Pearson -- funciona incluso con modelos no entrenados en L4S

pos_pool = np.where(labels == 1)[0]
eval_idx = rng.choice(pos_pool, min(40, len(pos_pool)), replace=False)

ratio_rn, ratio_eff = [], []
EPS = 1e-6

for i in eval_idx:
    patch  = patches_raw[i]
    mask_b = (masks_raw[i] > 0.1).astype(float)  # mascara binaria
    if mask_b.sum() < 10:   # muy pocos pixeles positivos -- omitir
        continue
    tensor, _ = preprocess_patch(patch)
    try:
        hm_rn  = get_gradcam(rn50,  tensor, TGT_RN50)
        hm_eff = get_gradcam(effb4, tensor, TGT_EFF)
        # Ratio = activacion media dentro / activacion media fuera
        for hm, store in [(hm_rn, ratio_rn), (hm_eff, ratio_eff)]:
            inside  = hm[mask_b > 0].mean()
            outside = hm[mask_b == 0].mean()
            if outside > EPS:
                store.append(inside / outside)
    except Exception:
        pass

print(f'ResNet-50       -- n validos: {len(ratio_rn):3d}')
print(f'EfficientNet-B4 -- n validos: {len(ratio_eff):3d}')

if len(ratio_rn) > 0:
    print(f'\nRatio Grad-CAM (dentro/fuera mascara):')
    print(f'  ResNet-50       : {np.mean(ratio_rn):.3f} +/- {np.std(ratio_rn):.3f}')
    print(f'  EfficientNet-B4 : {np.mean(ratio_eff):.3f} +/- {np.std(ratio_eff):.3f}')
    print(f'  Ratio > 1.0 = el modelo activa MAS dentro del deslizamiento que fuera')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
titles = ['ResNet-50', 'EfficientNet-B4']
colors = ['steelblue', 'darkorange']

for ax, ratios, label, color in zip(axes, [ratio_rn, ratio_eff], titles, colors):
    if len(ratios) > 1:
        ax.hist(ratios, bins=min(12, len(ratios)), color=color, edgecolor='white', alpha=0.85)
        ax.axvline(np.mean(ratios), color='red', linestyle='--',
                   label=f'media={np.mean(ratios):.2f}')
        ax.axvline(1.0, color='gray', linestyle=':', linewidth=1.5, label='ratio=1 (aleatorio)')
        ax.legend(fontsize=9)
        ax.set_xlabel('Activacion dentro / fuera de la mascara')
    else:
        ax.text(0.5, 0.5,
                'Sin datos\n(mascaras no encontradas\no todas uniformes)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=11, color='gray',
                bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.8))
        ax.set_xlabel('Activacion dentro / fuera de la mascara')
    ax.set_ylabel('Frecuencia')
    ax.set_title(label)

plt.suptitle('Discriminacion espacial Grad-CAM\n(ratio > 1 = activa mas en zona de deslizamiento)',
             fontsize=12)
plt.tight_layout()
out_corr = OUT_DIR / 'gradcam_discrimination.png'
plt.savefig(out_corr, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_corr}')


## 13 · Conclusiones e implicaciones para Colombia

### Interpretación de los Grad-CAM

| Aspecto | ResNet-50 | EfficientNet-B4 |
|---------|-----------|-----------------|
| Capa objetivo Grad-CAM | `layer4[-1]` — 4×4 receptive field | `conv_head` — mayor resolución |
| F1 global (L4S) | 0.784 | 0.756 |
| AUC-ROC global | ~0.813 | ~0.823 |

### Implicaciones para transferibilidad a Colombia

- Si el Grad-CAM se activa en zonas SAR/DEM (§11), el modelo es más robusto ante
  la alta nubosidad andina (60–80%).
- La correlación espacial heatmap↔máscara (§12) cuantifica la localización; valores
  altos sugieren que el modelo aprende la geometría del deslizamiento, no solo textura.
- **Fine-tuning recomendado**: con 100–200 parches de Antioquia, reentrenar
  `layer4` (ResNet-50) o los últimos 2 bloques (EfficientNet-B4).

### Trabajo futuro

- **Grad-CAM++** y **Score-CAM** — mayor precisión espacial que Grad-CAM estándar
- **Integrated Gradients** — atribución por canal individual (no solo espacial)
- Análisis sobre imágenes post-evento del **SGC/SIMMA** en Antioquia

---
*Notebook 3/3 — Transferibilidad a Colombia | Landslide4Sense — EAFIT 2026*  
*Notebooks relacionados: [01 Ablación](transferibilidad_01_ablacion_robustez.ipynb) | [02 Curvas](transferibilidad_02_curvas_rendimiento.ipynb)*